# Phase 2 Alt-Data Validation

Validates that the five national alt-data pipelines landed correctly:
1. **Census BPS vs FRED PERMIT** — permit counts should match within rounding
2. **FERC queue MW capacity by ISO over time** — snapshot trends
3. **USASpending construction obligations by month** — volume sanity check
4. **Ticker-mapped award totals** — compare against companies' reported government revenue

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()

DB_URL = (
    f"postgresql://{os.environ.get('POSTGRES_USER','urbangrowth')}"
    f":{os.environ.get('POSTGRES_PASSWORD','changeme')}"
    f"@{os.environ.get('POSTGRES_HOST','localhost')}"
    f":{os.environ.get('POSTGRES_PORT','5432')}"
    f"/{os.environ.get('POSTGRES_DB','urbangrowth')}"
)
engine = create_engine(DB_URL)

DATA_ROOT = Path(os.environ.get("URBANGROWTH_DATA_ROOT", "C:/urbangrowth_data"))

plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (12, 4)})
print("Setup complete")

---
## 1. Census BPS vs FRED PERMIT

Both report new privately-owned housing units authorized by building permits at monthly frequency.
- **BPS** (`census_bps` table, `jurisdiction_id = 'US'`) aggregates county-level survey data.
- **FRED PERMIT** (SA) is the headline release derived from the same BPS survey.

Expected: correlation > 0.99; mean absolute difference < 5%.

In [ ]:
bps_q = text("""
    SELECT period::date AS date, residential_permits
    FROM census_bps
    WHERE jurisdiction_id = 'US'
    ORDER BY period
""")

fred_q = text("""
    SELECT date::date, value
    FROM fred_series
    WHERE series_id = 'PERMIT'
    ORDER BY date
""")

with engine.connect() as conn:
    bps = pd.read_sql(bps_q, conn, parse_dates=["date"]).set_index("date")
    fred_permit = pd.read_sql(fred_q, conn, parse_dates=["date"]).set_index("date")

# FRED PERMIT is in thousands — scale to units
fred_permit["value"] = fred_permit["value"] * 1_000

combined = bps.join(fred_permit.rename(columns={"value": "fred_permit"}), how="inner")
combined.columns = ["bps_national", "fred_permit"]
print(f"Overlapping months: {len(combined)}")
combined.tail()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
combined["bps_national"].plot(ax=ax, label="BPS National", linewidth=1.5)
combined["fred_permit"].plot(ax=ax, label="FRED PERMIT (×1000)", linewidth=1.5, linestyle="--")
ax.set_title("BPS National Permits vs FRED PERMIT")
ax.set_ylabel("Units Authorized")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}K"))
ax.legend()

ax2 = axes[1]
ax2.scatter(combined["fred_permit"], combined["bps_national"], alpha=0.5, s=10)
lim_max = combined[["fred_permit", "bps_national"]].max().max() * 1.05
ax2.plot([0, lim_max], [0, lim_max], "r--", linewidth=1, label="y=x")
corr = combined.corr().iloc[0, 1]
ax2.set_title(f"Scatter (corr={corr:.4f})")
ax2.set_xlabel("FRED PERMIT")
ax2.set_ylabel("BPS National")
ax2.legend()

plt.tight_layout()
plt.show()

pct_diff = (combined["bps_national"] - combined["fred_permit"]).abs() / combined["fred_permit"] * 100
print(f"Correlation          : {corr:.6f}")
print(f"Mean abs % diff      : {pct_diff.mean():.2f}%")
print(f"Max abs % diff       : {pct_diff.max():.2f}%")
print(f"Pass (corr>0.99)     : {'YES' if corr > 0.99 else 'FAIL'}")
print(f"Pass (mean diff<5%)  : {'YES' if pct_diff.mean() < 5 else 'FAIL'}")

---
## 2. FERC Queue MW Capacity by ISO over Time

Each row in `ferc_queue` is a project in an ISO's interconnection queue at a given snapshot date.
Aggregating `capacity_mw` by ISO and snapshot gives a view of total queued capacity.

In [ ]:
ferc_q = text("""
    SELECT
        snapshot_date::date AS snapshot_date,
        iso,
        SUM(capacity_mw)    AS total_mw,
        COUNT(*)            AS project_count,
        SUM(CASE WHEN status = 'active' THEN capacity_mw ELSE 0 END) AS active_mw
    FROM ferc_queue
    GROUP BY snapshot_date, iso
    ORDER BY snapshot_date, iso
""")

with engine.connect() as conn:
    ferc = pd.read_sql(ferc_q, conn, parse_dates=["snapshot_date"])

print(f"Snapshots: {ferc['snapshot_date'].nunique()}")
print(f"ISOs     : {sorted(ferc['iso'].unique())}")
ferc.groupby("iso")[["total_mw", "active_mw", "project_count"]].mean().round(0)

In [ ]:
pivot = ferc.pivot_table(
    index="snapshot_date", columns="iso", values="total_mw", aggfunc="sum"
)

fig = px.line(
    pivot.reset_index().melt(id_vars="snapshot_date", var_name="iso", value_name="total_mw"),
    x="snapshot_date", y="total_mw", color="iso",
    title="FERC Queue: Total Queued Capacity (MW) by ISO",
    labels={"total_mw": "Capacity (MW)", "snapshot_date": "Snapshot Date"},
)
fig.update_layout(hovermode="x unified")
fig.show()

latest = ferc[ferc["snapshot_date"] == ferc["snapshot_date"].max()]
print("\nLatest snapshot summary:")
print(latest[["iso", "total_mw", "active_mw", "project_count"]].to_string(index=False))

In [ ]:
fuel_q = text("""
    SELECT
        iso,
        fuel_type,
        SUM(capacity_mw) AS mw
    FROM ferc_queue
    WHERE snapshot_date = (SELECT MAX(snapshot_date) FROM ferc_queue)
      AND status = 'active'
    GROUP BY iso, fuel_type
    ORDER BY iso, mw DESC
""")

with engine.connect() as conn:
    fuel = pd.read_sql(fuel_q, conn)

fig2 = px.bar(
    fuel, x="iso", y="mw", color="fuel_type", barmode="stack",
    title="Active Queue Capacity by ISO and Fuel Type (Latest Snapshot)",
    labels={"mw": "Capacity (MW)"},
)
fig2.show()

---
## 3. USASpending Construction Obligations by Month

Aggregates `amount` from `usaspending_awards` by `award_date` month to show federal construction spending cadence.

In [ ]:
usa_monthly_q = text("""
    SELECT
        DATE_TRUNC('month', award_date)::date AS month,
        SUM(amount)                            AS total_obligations,
        COUNT(*)                               AS award_count,
        COUNT(DISTINCT recipient_name)         AS unique_recipients
    FROM usaspending_awards
    GROUP BY 1
    ORDER BY 1
""")

with engine.connect() as conn:
    usa_monthly = pd.read_sql(usa_monthly_q, conn, parse_dates=["month"])

print(f"Months covered : {len(usa_monthly)}")
print(f"Total awards   : {usa_monthly['award_count'].sum():,.0f}")
print(f"Total obligated: ${usa_monthly['total_obligations'].sum()/1e9:.1f}B")
usa_monthly.tail()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

ax = axes[0]
ax.bar(usa_monthly["month"], usa_monthly["total_obligations"] / 1e9, width=25, color="steelblue", alpha=0.8)
ax.set_title("Federal Construction Contract Awards — Monthly Obligations")
ax.set_ylabel("$ Billions")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.1f}B"))

ax2 = axes[1]
ax2.bar(usa_monthly["month"], usa_monthly["award_count"], width=25, color="darkorange", alpha=0.8)
ax2.set_title("Award Count per Month")
ax2.set_ylabel("Number of Awards")
ax2.set_xlabel("Month")

plt.tight_layout()
plt.show()

usa_monthly["cal_month"] = usa_monthly["month"].dt.month
seasonal = usa_monthly.groupby("cal_month")["total_obligations"].mean() / 1e9
print("\nMean monthly obligation by calendar month ($B):")
print(seasonal.to_string())

---
## 4. Ticker-Mapped Award Totals vs Reported Government Revenue

Cross-references `recipient_ticker_guess` in `usaspending_awards` against known rough government revenue percentages from public filings.
This is a sanity check — exact match not expected (USASpending covers federal only; companies may have state contracts).

In [ ]:
ticker_q = text("""
    SELECT
        recipient_ticker_guess          AS ticker,
        COUNT(*)                        AS award_count,
        SUM(amount)                     AS total_awarded,
        MIN(award_date)::date           AS first_award,
        MAX(award_date)::date           AS last_award
    FROM usaspending_awards
    WHERE recipient_ticker_guess IS NOT NULL
    GROUP BY recipient_ticker_guess
    ORDER BY total_awarded DESC
""")

with engine.connect() as conn:
    by_ticker = pd.read_sql(ticker_q, conn)

print(f"Tickers matched: {len(by_ticker)}")
print(f"Awards matched : {by_ticker['award_count'].sum():,}")
print(f"Total matched  : ${by_ticker['total_awarded'].sum()/1e9:.1f}B")
by_ticker

In [ ]:
unmatched_q = text("""
    SELECT
        recipient_name,
        COUNT(*)        AS awards,
        SUM(amount)     AS total,
        MIN(award_date) AS first,
        MAX(award_date) AS last
    FROM usaspending_awards
    WHERE recipient_ticker_guess IS NULL
    GROUP BY recipient_name
    HAVING SUM(amount) > 10000000
    ORDER BY total DESC
    LIMIT 30
""")

with engine.connect() as conn:
    unmatched = pd.read_sql(unmatched_q, conn)

print(f"Unmatched recipients with >$10M total: {len(unmatched)}")
print()
print(
    unmatched[["recipient_name", "awards", "total"]]
    .assign(total_M=lambda df: (df["total"] / 1e6).round(1))
    [["recipient_name", "awards", "total_M"]]
    .to_string(index=False)
)

In [ ]:
# Known approximate government revenue (from 10-K / investor presentations, 2024)
# Format: ticker -> (gov_rev_pct, annual_rev_B)
known_gov_rev = {
    "PWR":  (0.05, 22.0),   # Quanta ~5% federal
    "EME":  (0.40, 14.0),   # EMCOR ~40% government
    "FIX":  (0.10,  5.0),   # Comfort Systems ~10%
    "J":    (0.20, 16.0),   # Jacobs ~20% government
    "ACM":  (0.25, 16.0),   # AECOM ~25% government
    "KBR":  (0.35,  7.0),   # KBR ~35% government
    "STRL": (0.08,  2.0),   # Sterling Construction ~8%
}

rows = []
for ticker, (pct, rev_B) in known_gov_rev.items():
    matched = by_ticker[by_ticker["ticker"] == ticker]
    awarded_B = float(matched["total_awarded"].sum()) / 1e9 if len(matched) else 0.0
    years_of_data = (
        (matched["last_award"].max() - matched["first_award"].min()).days / 365
        if len(matched) and pd.notna(matched["first_award"].min()) else 1
    )
    annual_B = awarded_B / max(years_of_data, 1)
    implied_B = rev_B * pct
    rows.append({
        "ticker":              ticker,
        "filed_gov_rev_B":     round(implied_B, 2),
        "usaspending_annual_B": round(annual_B, 2),
        "ratio":               round(annual_B / implied_B, 2) if implied_B else None,
    })

sanity = pd.DataFrame(rows)
print("Sanity check: USASpending annualised vs filed government revenue ($B)")
print("Expected ratio 0.2–0.9 (federal subset of total gov rev)")
print()
print(sanity.to_string(index=False))

In [ ]:
fig = px.bar(
    by_ticker.head(15),
    x="ticker", y="total_awarded",
    title="USASpending Total Awards by Matched Ticker (Full History)",
    labels={"total_awarded": "Total Awarded ($)", "ticker": "Ticker"},
    text=by_ticker.head(15)["total_awarded"].apply(lambda x: f"${x/1e9:.1f}B"),
)
fig.update_traces(textposition="outside")
fig.update_yaxes(tickformat="$.0s")
fig.show()

---
## 5. DOT TIPs — Cache and DB Check

DOT TIP data is scraped from state agency portals (TxDOT, Caltrans, ADOT). This cell confirms parquet
caches exist and the `dot_tips` DB table has rows.

In [ ]:
from datetime import date

today = date.today().isoformat()
dot_caches = [
    DATA_ROOT / "raw" / "dot_tips" / "TX" / f"tip_{today}.parquet",
    DATA_ROOT / "raw" / "dot_tips" / "CA" / f"tip_{today}.parquet",
    DATA_ROOT / "raw" / "dot_tips" / "AZ" / f"tip_{today}.parquet",
]

for p in dot_caches:
    status = "FOUND" if p.exists() else "MISSING — run: ug ingest dot-tips"
    print(f"{status:45s} {p.name}")

dot_count_q = text("""
    SELECT state, COUNT(*) AS rows, SUM(total_cost) AS total_cost
    FROM dot_tips
    GROUP BY state
    ORDER BY state
""")
with engine.connect() as conn:
    dot_summary = pd.read_sql(dot_count_q, conn)

if dot_summary.empty:
    print("\ndot_tips table is empty — run: ug ingest dot-tips")
else:
    print("\ndot_tips DB summary:")
    print(dot_summary.to_string(index=False))

---
## 6. FRED Wide Parquet — Shape and Coverage Check

In [ ]:
fred_parquet = DATA_ROOT / "raw" / "fred" / "fred_series_wide.parquet"

if fred_parquet.exists():
    wide = pd.read_parquet(fred_parquet)
    print(f"Shape     : {wide.shape}")
    print(f"Date range: {wide.index.min().date()} — {wide.index.max().date()}")
    print("\nNull counts per series:")
    print(wide.isnull().sum().to_string())
    print("\nSample (last 3 months):")
    display(wide.tail(3).T)
else:
    print(f"Parquet not found at {fred_parquet}")
    print("Run: ug ingest fred")

---
## 7. Census BPS — Coverage by Geography Level

In [ ]:
geo_q = text("""
    SELECT
        CASE
            WHEN jurisdiction_id = 'US'                                    THEN 'national'
            WHEN LENGTH(jurisdiction_id) = 5 AND jurisdiction_id ~ '^[0-9]+$' THEN 'county'
            ELSE 'msa'
        END                                  AS geo_level,
        COUNT(DISTINCT jurisdiction_id)      AS jurisdiction_count,
        COUNT(*)                             AS row_count,
        MIN(period)::date                    AS min_date,
        MAX(period)::date                    AS max_date
    FROM census_bps
    GROUP BY 1
    ORDER BY 1
""")

with engine.connect() as conn:
    geo = pd.read_sql(geo_q, conn)
print(geo.to_string(index=False))

# Phoenix (38060) and Austin (12420) CBSA permit totals
cbsa_q = text("""
    SELECT jurisdiction_id, SUM(residential_permits) AS total_permits, COUNT(*) AS months
    FROM census_bps
    WHERE jurisdiction_id IN ('38060', '12420')
    GROUP BY jurisdiction_id
""")
with engine.connect() as conn:
    cbsa = pd.read_sql(cbsa_q, conn)

cbsa["city"] = cbsa["jurisdiction_id"].map({"38060": "Phoenix", "12420": "Austin"})
print("\nPhoenix / Austin permit coverage:")
print(cbsa[["city", "jurisdiction_id", "months", "total_permits"]].to_string(index=False))